In [7]:
import re

class ASTNode:
    pass

class WhileNode(ASTNode):
    def __init__(self, condition, body):
        self.condition = condition  # ConditionNode
        self.body = body            # AssignmentNode

class ConditionNode(ASTNode):
    def __init__(self, left, op, right):
        self.left = left
        self.op = op
        self.right = right

class AssignmentNode(ASTNode):
    def __init__(self, variable, expression):
        self.variable = variable
        self.expression = expression

class WhileCompiler:
    def __init__(self, source_code):
        self.source_code = source_code
        self.label_counter = 1
        self.symbol_table = {"count": "int", "x": "int"} # Simpan variabel pra-definisi untuk cek semantik

    def new_label(self):
        lbl = f"L{self.label_counter}"
        self.label_counter += 1
        return lbl

    # --- TAHAP 1: ANALISIS LEKSIKAL ---
    def lexical_analysis(self):
        # Memisahkan token dengan memberikan spasi pada karakter spesial
        spaced_code = (self.source_code
                       .replace('(', ' ( ')
                       .replace(')', ' ) ')
                       .replace('{', ' { ')
                       .replace('}', ' } ')
                       .replace('<', ' < ')
                       .replace('>', ' > ')
                       .replace('=', ' = '))

        # Split berdasarkan spasi dan bersihkan token kosong
        tokens = [t for t in spaced_code.split() if t]
        return tokens

    # --- TAHAP 2: ANALISIS SINTAKSIS (Parsing ke AST) ---
    def syntax_analysis(self, tokens):
        try:
            if tokens[0] != 'while':
                raise SyntaxError("Diharapkan keyword 'while'")

            if tokens[1] != '(':
                raise SyntaxError("Diharapkan '(' setelah 'while'")

            # Parsing Kondisi
            idx_close_paren = tokens.index(')')
            cond_tokens = tokens[2:idx_close_paren]
            if len(cond_tokens) != 3:
                raise SyntaxError("Struktur kondisi tidak valid. Contoh benar: (x < 10)")
            cond_node = ConditionNode(cond_tokens[0], cond_tokens[1], cond_tokens[2])

            if tokens[idx_close_paren + 1] != '{':
                raise SyntaxError("Diharapkan '{' untuk memulai blok body")

            # Parsing Body (Assignment sederhana)
            idx_open_brace = idx_close_paren + 1
            idx_close_brace = tokens.index('}')
            body_tokens = tokens[idx_open_brace + 1 : idx_close_brace]

            if '=' not in body_tokens or len(body_tokens) != 3:
                raise SyntaxError("Struktur body harus berupa assignment tunggal. Contoh: x = 5")

            body_node = AssignmentNode(body_tokens[0], body_tokens[2])

            # Gabungkan ke root node AST
            return WhileNode(cond_node, body_node)

        except (ValueError, IndexError):
            raise SyntaxError("Struktur sintaksis 'while' tidak lengkap atau salah penutupan bracket.")

    # --- TAHAP 3: ANALISIS SEMANTIK ---
    def semantic_analysis(self, ast):
        # Aturan Semantik 1: Cek apakah variabel di dalam kondisi sudah dideklarasikan
        left_var = ast.condition.left
        if left_var not in self.symbol_table:
            raise NameError(self.format_semantic_error(f"Variabel '{left_var}' belum dideklarasikan!"))

        # Aturan Semantik 2: Cek apakah variabel target assignment sudah dideklarasikan
        target_var = ast.body.variable
        if target_var not in self.symbol_table:
            raise NameError(self.format_semantic_error(f"Variabel '{target_var}' belum dideklarasikan!"))

        # Aturan Semantik 3: Validasi tipe data sederhana (contoh: memastikan pembanding adalah angka jika variabel bertipe int)
        if self.symbol_table[left_var] == "int":
            if not ast.condition.right.isdigit() and ast.condition.right not in self.symbol_table:
                raise TypeError(self.format_semantic_error(f"Kesalahan Tipe: '{left_var}' (int) tidak bisa dibandingkan dengan '{ast.condition.right}'"))

        return True

    def format_semantic_error(self, message):
        return f"\033[91m[Semantic Error] {message}\033[0m"

    # --- TAHAP 4: GENERASI KODE (TAC) ---
    def generate_tac(self, ast):
        label_start = self.new_label()
        label_body = self.new_label()
        label_end = self.new_label()

        cond = ast.condition
        body = ast.body

        tac = []
        tac.append(f"{label_start}:")
        # Jika kondisi terpenuhi, lompat ke isi perulangan (label_body)
        tac.append(f"if {cond.left} {cond.op} {cond.right} goto {label_body}")
        # Jika gagal, lompat ke akhir perulangan
        tac.append(f"goto {label_end}")

        # Blok Isi (Body) Loop
        tac.append(f"{label_body}:")
        tac.append(f"{body.variable} = {body.expression}")
        # Kembali cek kondisi awal
        tac.append(f"goto {label_start}")

        # Label Penutup
        tac.append(f"{label_end}:")

        return "\n".join(tac)

    # --- FUNGSI UTAMA PIPELINE COMPILER ---
    def compile(self):
        print(f"Source Code: {self.source_code}\n")

        # 1. Leksikal
        tokens = self.lexical_analysis()
        print("--- 1. TAHAP ANALISIS LEKSIKAL (Tokens) ---")
        print(tokens)
        print("-" * 45)

        # 2. Sintaksis
        ast = self.syntax_analysis(tokens)
        print("--- 2. TAHAP ANALISIS SINTAKSIS (AST Terbentuk) ---")
        print(f"Root Node: WhileNode")
        print(f" ├── Condition: {ast.condition.left} {ast.condition.op} {ast.condition.right}")
        print(f" └── Body Loop: {ast.body.variable} = {ast.body.expression}")
        print("-" * 45)

        # 3. Semantik
        print("--- 3. TAHAP ANALISIS SEMANTIK ---")
        if self.semantic_analysis(ast):
            print("Hasil: Validasi Semantik Sukses (Semua variabel terdaftar & Tipe data cocok).")
        print("-" * 45)

        # 4. TAC Generation
        tac_code = self.generate_tac(ast)
        print("--- 4. THREE-ADDRESS CODE (TAC) GENERATION ---")
        print(tac_code)
        print("-" * 45)




In [8]:
if __name__ == "__main__":
    # Test Case 1: Kode Sukses
    source_program = "while ( count < 5 ) { x = count }"
    compiler = WhileCompiler(source_program)
    compiler.compile()

    print("\n\n")

    # Test Case 2: Simulasi Error Semantik (Memicu error variabel belum dideklarasikan)
    try:
        source_error = "while ( total < 5 ) { x = 1 }" # 'total' tidak ada di symbol_table
        compiler_err = WhileCompiler(source_error)
        compiler_err.compile()
    except Exception as e:
        print(e)

Source Code: while ( count < 5 ) { x = count }

--- 1. TAHAP ANALISIS LEKSIKAL (Tokens) ---
['while', '(', 'count', '<', '5', ')', '{', 'x', '=', 'count', '}']
---------------------------------------------
--- 2. TAHAP ANALISIS SINTAKSIS (AST Terbentuk) ---
Root Node: WhileNode
 ├── Condition: count < 5
 └── Body Loop: x = count
---------------------------------------------
--- 3. TAHAP ANALISIS SEMANTIK ---
Hasil: Validasi Semantik Sukses (Semua variabel terdaftar & Tipe data cocok).
---------------------------------------------
--- 4. THREE-ADDRESS CODE (TAC) GENERATION ---
L1:
if count < 5 goto L2
goto L3
L2:
x = count
goto L1
L3:
---------------------------------------------



Source Code: while ( total < 5 ) { x = 1 }

--- 1. TAHAP ANALISIS LEKSIKAL (Tokens) ---
['while', '(', 'total', '<', '5', ')', '{', 'x', '=', '1', '}']
---------------------------------------------
--- 2. TAHAP ANALISIS SINTAKSIS (AST Terbentuk) ---
Root Node: WhileNode
 ├── Condition: total < 5
 └── Body Lo